# 01 — Data Exploration

**Goal:** understand the supplied cold-chain logistics dataset before making assumptions or building models.

Run from the project root. This notebook does not modify the raw CSV.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists() and (ROOT.parent / "data" / "raw").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "raw" / "dynamic_supply_chain_logistics_dataset.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Open this notebook from the project root "
        "or update DATA_PATH to the CSV location."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)

Dataset shape: (32065, 26)


## 1. Preview and schema

In [2]:
display(df.head())
display(df.dtypes.to_frame("dtype"))
print("Columns:", df.columns.tolist())

,timestamp,vehicle_gps_latitude,vehicle_gps_longitude,fuel_consumption_rate,eta_variation_hours,traffic_congestion_level,warehouse_inventory_level,loading_unloading_time,handling_equipment_availability,order_fulfillment_status,...,iot_temperature,cargo_condition_status,route_risk_level,customs_clearance_time,driver_behavior_score,fatigue_monitoring_score,disruption_likelihood_score,delay_probability,risk_classification,delivery_time_deviation
0,2021-01-01 00:00:00,40.375568,-77.014318,5.136512,4.998009,5.927586,985.716862,4.951392,0.481294,0.761166,...,0.574400,0.777263,1.182116,0.502006,0.033843,0.978599,0.506152,0.885291,Moderate Risk,9.110682
1,2021-01-01 01:00:00,33.507818,-117.036902,5.101512,0.984929,1.591992,396.700206,1.030379,0.620780,0.196594,...,-9.753493,0.091839,9.611988,0.966774,0.201725,0.918586,0.980784,0.544178,High Risk,8.175281
2,2021-01-01 02:00:00,30.020640,-75.269224,5.090803,4.972665,8.787765,832.408935,4.220229,0.810933,0.152742,...,-6.491034,0.253529,6.570431,0.945627,0.264045,0.394215,0.998633,0.803322,High Risk,1.283594
3,2021-01-01 03:00:00,36.649223,-70.190529,8.219558,3.095064,0.045257,0.573283,0.530186,0.008525,0.811885,...,-0.151276,0.877576,0.548952,4.674035,0.362885,0.905444,0.993320,0.025977,High Risk,9.304897
4,2021-01-01 04:00:00,30.001279,-70.012195,5.000075,3.216077,8.004851,914.925067,3.620890,0.020083,0.053659,...,2.429448,0.262081,8.861443,3.445429,0.016957,0.258702,0.912433,0.991122,High Risk,7.752484


,dtype
timestamp,str
vehicle_gps_latitude,float64
vehicle_gps_longitude,float64
fuel_consumption_rate,float64
eta_variation_hours,float64
traffic_congestion_level,float64
warehouse_inventory_level,float64
loading_unloading_time,float64
handling_equipment_availability,float64
order_fulfillment_status,float64


Columns: ['timestamp', 'vehicle_gps_latitude', 'vehicle_gps_longitude', 'fuel_consumption_rate', 'eta_variation_hours', 'traffic_congestion_level', 'warehouse_inventory_level', 'loading_unloading_time', 'handling_equipment_availability', 'order_fulfillment_status', 'weather_condition_severity', 'port_congestion_level', 'shipping_costs', 'supplier_reliability_score', 'lead_time_days', 'historical_demand', 'iot_temperature', 'cargo_condition_status', 'route_risk_level', 'customs_clearance_time', 'driver_behavior_score', 'fatigue_monitoring_score', 'disruption_likelihood_score', 'delay_probability', 'risk_classification', 'delivery_time_deviation']


## 2. Dataset health

In [3]:
quality = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
})
display(quality.sort_values("missing_values", ascending=False))
print("Duplicate rows:", df.duplicated().sum())

,missing_values,missing_percent,unique_values
timestamp,0,0.0,32065
vehicle_gps_latitude,0,0.0,32065
vehicle_gps_longitude,0,0.0,32065
fuel_consumption_rate,0,0.0,32037
eta_variation_hours,0,0.0,32065
traffic_congestion_level,0,0.0,32065
warehouse_inventory_level,0,0.0,32065
loading_unloading_time,0,0.0,32065
handling_equipment_availability,0,0.0,32065
order_fulfillment_status,0,0.0,32065


Duplicate rows: 0


## 3. Numeric summary

In [4]:
display(df.describe().T.round(3))

,count,mean,std,min,25%,50%,75%,max
vehicle_gps_latitude,32065.0,38.024,6.918,30.000,31.281,36.414,44.454,50.000
vehicle_gps_longitude,32065.0,-90.117,17.369,-120.000,-106.254,-86.293,-73.079,-70.000
fuel_consumption_rate,32065.0,8.012,4.265,5.000,5.020,5.636,9.670,20.000
eta_variation_hours,32065.0,2.893,2.274,-2.000,1.186,3.882,4.884,5.000
traffic_congestion_level,32065.0,4.991,3.532,0.000,1.475,4.981,8.535,10.000
warehouse_inventory_level,32065.0,299.255,323.444,0.000,16.052,157.288,540.598,999.999
loading_unloading_time,32065.0,2.292,1.554,0.500,0.775,1.917,3.734,5.000
handling_equipment_availability,32065.0,0.303,0.326,0.000,0.017,0.160,0.554,1.000
order_fulfillment_status,32065.0,0.601,0.346,0.000,0.277,0.681,0.938,1.000
weather_condition_severity,32065.0,0.498,0.353,0.000,0.144,0.496,0.850,1.000


## 4. Time coverage

In [5]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
print("Start:", df["timestamp"].min())
print("End:  ", df["timestamp"].max())
print("Invalid timestamps:", df["timestamp"].isna().sum())
display(df["timestamp"].dt.year.value_counts().sort_index().rename("records").to_frame())

Start: 2021-01-01 00:00:00
End:   2024-08-29 00:00:00
Invalid timestamps: 0


,records
timestamp,
2021,8760
2022,8760
2023,8760
2024,5785


## 5. Important categorical distributions

In [6]:
for col in ["order_fulfillment_status", "cargo_condition_status", "route_risk_level", "risk_classification", "weather_condition_severity"]:
    if col in df.columns:
        print(f"\\n{col}")
        display(df[col].value_counts(dropna=False).to_frame("count"))

\norder_fulfillment_status


,count
order_fulfillment_status,
0.761166,1
0.196594,1
0.152742,1
0.811885,1
0.053659,1
...,...
0.992997,1
0.946031,1
0.437687,1


\ncargo_condition_status


,count
cargo_condition_status,
0.777263,1
0.091839,1
0.253529,1
0.877576,1
0.262081,1
...,...
0.199377,1
0.673775,1
0.949644,1


\nroute_risk_level


,count
route_risk_level,
1.182116,1
9.611988,1
6.570431,1
0.548952,1
8.861443,1
...,...
9.893402,1
0.854378,1
4.480552,1


\nrisk_classification


,count
risk_classification,
High Risk,23944
Moderate Risk,5011
Low Risk,3110


\nweather_condition_severity


,count
weather_condition_severity,
0.359066,1
0.230660,1
0.027210,1
0.616619,1
0.952385,1
...,...
0.792648,1
0.034156,1
0.022338,1
